In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ultralytics] [ultralytics]


In [ ]:
import os
import copy
import yaml
import torch

from pathlib import Path
from ultralytics import YOLO

print("=" * 70)
print("YOLOv9t — VISDRONE CONTROLLED EXPERIMENT")
print("=" * 70)

print("Ultralytics:", __import__("ultralytics").__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
DATASET_ROOT = Path("/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final")

print("Dataset exists:", DATASET_ROOT.exists())

if DATASET_ROOT.exists():
    print("Dataset:", DATASET_ROOT)
else:
    raise FileNotFoundError(
        "VisDrone_Final is not available in /kaggle/working. "
        "Attach/copy the corrected VisDrone dataset first."
    )

In [ ]:
DATA_YAML = Path("/kaggle/working/visdrone_final.yaml")

CLASS_NAMES = [
    "pedestrian",
    "people",
    "bicycle",
    "car",
    "van",
    "truck",
    "tricycle",
    "awning-tricycle",
    "bus",
    "motor"
]

dataset_cfg = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 10,
    "names": CLASS_NAMES
}

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(dataset_cfg, f, sort_keys=False)

print()
print("=" * 70)
print("DATASET YAML")
print("=" * 70)
print(DATA_YAML)
print(yaml.safe_dump(dataset_cfg, sort_keys=False))

In [ ]:
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

assert cfg["nc"] == 10
assert len(cfg["names"]) == 10

for split in ["train", "val", "test"]:
    p = Path(cfg["path"]) / cfg[split]
    print(f"{split:5s}: {p} -> {'✓' if p.exists() else '✗'}")

print("\n✓ Dataset configuration verified")

In [ ]:
MODEL_YAML = "/kaggle/working/yolov9t_visdrone_10class.yaml"

base_model = YOLO("yolov9t.yaml")

model_yaml = copy.deepcopy(base_model.model.yaml)

model_yaml["nc"] = 10
model_yaml["names"] = CLASS_NAMES

with open(MODEL_YAML, "w") as f:
    yaml.safe_dump(model_yaml, f, sort_keys=False)

model = YOLO(MODEL_YAML)

architecture_params = sum(
    p.numel() for p in model.model.parameters()
)

print("=" * 70)
print("YOLOv9t 10-CLASS ARCHITECTURE")
print("=" * 70)

print("Model YAML:", MODEL_YAML)
print("Parameters:", f"{architecture_params:,}")

print("\n✓ 10-class YOLOv9t architecture created")

In [ ]:
print("=" * 70)
print("TRANSFERRING YOLOv9t PRETRAINED WEIGHTS")
print("=" * 70)

pretrained = YOLO("yolov9t.pt")

source_state = pretrained.model.state_dict()
target_state = model.model.state_dict()

copied = 0
skipped = 0

for key, source_tensor in source_state.items():

    if key in target_state:

        if target_state[key].shape == source_tensor.shape:
            target_state[key] = source_tensor.clone()
            copied += 1
        else:
            skipped += 1

    else:
        skipped += 1

model.model.load_state_dict(
    target_state,
    strict=False
)

final_params = sum(
    p.numel() for p in model.model.parameters()
)

print()
print("Compatible tensors copied :", copied)
print("Skipped tensors           :", skipped)
print("Final parameters          :", f"{final_params:,}")

print()
print("✓ Compatible YOLOv9t weights transferred")
print("✓ 10-class detection head retained")

In [ ]:
print("=" * 70)
print("YOLOv9t ARCHITECTURE VERIFICATION")
print("=" * 70)

verified_params = sum(
    p.numel() for p in model.model.parameters()
)

print("Total parameters:", f"{verified_params:,}")

assert verified_params == final_params

print("✓ Parameter count verified")

In [ ]:
device = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

model.model.to(device)
model.model.eval()

dummy = torch.randn(
    1, 3, 640, 640,
    device=device
)

with torch.no_grad():
    output = model.model(dummy)

print("=" * 70)
print("FORWARD PASS")
print("=" * 70)

print("Input shape :", tuple(dummy.shape))
print("Forward pass: SUCCESS")

In [ ]:
print("=" * 70)
print("STARTING YOLOv9t CONTROLLED TRAINING")
print("=" * 70)

results = model.train(
    data="/kaggle/working/visdrone_final.yaml",

    epochs=100,
    imgsz=640,
    batch=8,
    device=0,

    # Fixed optimizer protocol
    optimizer="AdamW",
    lr0=0.002,
    weight_decay=0.0005,

    # Fixed scheduler
    cos_lr=True,
    warmup_epochs=3,

    # Reproducibility
    amp=True,
    seed=42,
    deterministic=True,

    # Validation
    patience=20,
    max_det=1000,

    # Data loading
    workers=2,

    # No special augmentation
    copy_paste=0.0,

    # Pretrained weights already transferred manually
    pretrained=False,

    project="/kaggle/working/YOLO_VisDrone_Benchmark",
    name="YOLOv9t_AdamW_seed42",
    exist_ok=False
)

print()
print("=" * 70)
print("YOLOv9t TRAINING COMPLETED")
print("=" * 70)

In [ ]:
copy_paste=0.0

In [ ]:
BEST = Path(
    "/kaggle/working/YOLO_VisDrone_Benchmark/"
    "YOLOv9t_AdamW_seed42/weights/best.pt"
)

assert BEST.exists(), f"Best checkpoint not found: {BEST}"

best_model = YOLO(str(BEST))

print("=" * 70)
print("YOLOv9t INDEPENDENT VALIDATION")
print("=" * 70)

metrics = best_model.val(
    data="/kaggle/working/visdrone_final.yaml",
    imgsz=640,
    batch=8,
    device=0,
    max_det=1000,
    workers=2
)

print()
print(f"Precision   : {metrics.box.mp:.5f}")
print(f"Recall      : {metrics.box.mr:.5f}")
print(f"mAP50       : {metrics.box.map50:.5f}")
print(f"mAP50-95    : {metrics.box.map:.5f}")

In [2]:
from ultralytics import YOLO
runs = {  # name: path to best.pt
    "YOLOv9t": "/kaggle/input/notebooks/narendramodigari/yolov9t-drone-yolo/YOLO_VisDrone_Benchmark/YOLOv9t_AdamW_seed42/weights/best.pt",
    # add the other models here
}
for name, w in runs.items():
    YOLO(w).val(data="visdrone_final.yaml", split="test", imgsz=640,
                conf=0.001, iou=0.7, max_det=500, save_json=True, name=name)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLOv9t_visdrone_10class summary (fused): 197 layers, 1,972,734 parameters, 0 gradients, 7.6 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 2.4±1.1 MB/s, size: 182.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels... 1610 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1610/1610 19.4it/s 1:230.1sss
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/narendramodigari/visdrone-label-correc

In [4]:
# ============================================================
# SIZE-STRATIFIED AP / RECALL EVALUATION
# ============================================================

!pip install -q pycocotools pillow

import argparse
import csv
import json
from pathlib import Path

import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval


BINS = {
    "coco": [
        ("small", 0, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],

    "fine": [
        ("tiny", 0, 16**2),
        ("small", 16**2, 32**2),
        ("medium", 32**2, 96**2),
        ("large", 96**2, 1e10)
    ],
}


def build_gt(img_dir, lbl_dir, n_classes, cat_offset):

    images = []
    anns = []
    aid = 1

    for p in sorted(Path(img_dir).iterdir()):

        if p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        w, h = Image.open(p).size

        iid = int(p.stem) if p.stem.isnumeric() else p.stem

        images.append(
            dict(
                id=iid,
                width=w,
                height=h,
                file_name=p.name
            )
        )

        lf = Path(lbl_dir) / (p.stem + ".txt")

        if not lf.exists():
            continue

        for line in lf.read_text().strip().splitlines():

            parts = line.split()

            if len(parts) < 5:
                continue

            c = int(float(parts[0]))

            xc, yc, bw, bh = (
                float(v) for v in parts[1:5]
            )

            # Convert normalized YOLO coordinates
            # to ORIGINAL image pixel coordinates
            bw = bw * w
            bh = bh * h

            x = xc * w - bw / 2
            y = yc * h - bh / 2

            anns.append(
                dict(
                    id=aid,
                    image_id=iid,
                    category_id=c + cat_offset,
                    bbox=[x, y, bw, bh],
                    area=bw * bh,
                    iscrowd=0
                )
            )

            aid += 1

    cats = [
        dict(
            id=i + cat_offset,
            name=str(i)
        )
        for i in range(n_classes)
    ]

    gt = COCO()

    gt.dataset = dict(
        images=images,
        annotations=anns,
        categories=cats
    )

    gt.createIndex()

    return gt


def mean_valid(a):

    a = a[a > -1]

    return float(a.mean()) if a.size else float("nan")


def evaluate(gt, pred_file, bins, maxdet):

    with open(pred_file, "r") as f:
        preds = json.load(f)

    dt = gt.loadRes(preds)

    E = COCOeval(
        gt,
        dt,
        "bbox"
    )

    E.params.maxDets = [10, 100, maxdet]

    E.params.areaRng = (
        [[0, 1e10]]
        + [[lo, hi] for _, lo, hi in bins]
    )

    E.params.areaRngLbl = (
        ["all"]
        + [b[0] for b in bins]
    )

    E.evaluate()
    E.accumulate()

    prec = E.eval["precision"]
    rec = E.eval["recall"]

    # maxDet index
    m = 2

    rows = []

    areas = np.array([
        a["area"]
        for a in gt.dataset["annotations"]
    ])

    for a, lbl in enumerate(E.params.areaRngLbl):

        lo, hi = E.params.areaRng[a]

        n = int(
            ((areas >= lo) & (areas <= hi)).sum()
        )

        rows.append(
            dict(
                bin=lbl,
                n_gt=n,

                AP50_95=mean_valid(
                    prec[:, :, :, a, m]
                ),

                AP50=mean_valid(
                    prec[0, :, :, a, m]
                ),

                AR50_95=mean_valid(
                    rec[:, :, a, m]
                ),

                AR50=mean_valid(
                    rec[0, :, a, m]
                )
            )
        )

    return rows


def main(argv=None):

    ap = argparse.ArgumentParser()

    ap.add_argument(
        "--images",
        required=True
    )

    ap.add_argument(
        "--labels",
        required=True
    )

    ap.add_argument(
        "--runs",
        nargs="+",
        required=True,
        help="NAME=path/to/predictions.json"
    )

    ap.add_argument(
        "--nc",
        type=int,
        default=10
    )

    ap.add_argument(
        "--bins",
        choices=BINS,
        default="fine"
    )

    ap.add_argument(
        "--maxdet",
        type=int,
        default=500
    )

    ap.add_argument(
        "--cat_offset",
        type=int,
        default=-1,
        help="0 or 1; -1 = infer from predictions"
    )

    ap.add_argument(
        "--out",
        default="size_results.csv"
    )

    a = ap.parse_args(argv)

    runs = dict(
        r.split("=", 1)
        for r in a.runs
    )

    # --------------------------------------------------------
    # Infer category offset
    # --------------------------------------------------------

    off = a.cat_offset

    if off < 0:

        first_file = next(
            iter(runs.values())
        )

        with open(first_file, "r") as f:
            first = json.load(f)

        off = min(
            p["category_id"]
            for p in first
        )

        print(
            "Inferred category offset:",
            off
        )

    # --------------------------------------------------------
    # Build ground truth
    # --------------------------------------------------------

    gt = build_gt(
        a.images,
        a.labels,
        a.nc,
        off
    )

    print(
        f"GT: {len(gt.dataset['images'])} images, "
        f"{len(gt.dataset['annotations'])} boxes"
    )

    # --------------------------------------------------------
    # Evaluate models
    # --------------------------------------------------------

    out = []

    for name, f in runs.items():

        print()
        print("=" * 70)
        print(f"Evaluating: {name}")
        print(f"Prediction file: {f}")
        print("=" * 70)

        for r in evaluate(
            gt,
            f,
            BINS[a.bins],
            a.maxdet
        ):

            r["model"] = name

            out.append(r)

            print(
                f"{name:12s} "
                f"{r['bin']:7s} "
                f"n={r['n_gt']:6d} "
                f"AP50={r['AP50']:.3f} "
                f"AP50-95={r['AP50_95']:.3f} "
                f"AR50={r['AR50']:.3f} "
                f"AR50-95={r['AR50_95']:.3f}"
            )

    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    with open(
        a.out,
        "w",
        newline=""
    ) as fh:

        w = csv.DictWriter(
            fh,
            fieldnames=[
                "model",
                "bin",
                "n_gt",
                "AP50",
                "AP50_95",
                "AR50",
                "AR50_95"
            ]
        )

        w.writeheader()
        w.writerows(out)

    print()
    print("=" * 70)
    print("Saved:")
    print(a.out)
    print("=" * 70)


# ============================================================
# KAGGLE CONFIGURATION
# ============================================================

main([
    "--images",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/images",

    "--labels",
    "/kaggle/input/datasets/narendramodigari/visdrone-label-corrected/VisDrone_Final/test/labels",

    "--runs",

    "YOLOv9t=/kaggle/working/runs/detect/YOLOv9t/predictions.json",

    # "YOLOv9t=/kaggle/working/runs/detect/v9t/predictions.json",

    "--nc",
    "10",

    "--bins",
    "fine",

    "--maxdet",
    "500",

    "--cat_offset",
    "-1",

    "--out",
    "/kaggle/working/size_results.csv"
])

Inferred category offset: 1
creating index...
index created!
GT: 1610 images, 75102 boxes

Evaluating: YOLOv9t
Prediction file: /kaggle/working/runs/detect/YOLOv9t/predictions.json
Loading and preparing results...
DONE (t=3.65s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=213.61s).
Accumulating evaluation results...
DONE (t=13.58s).
YOLOv9t      all     n= 75102 AP50=0.238 AP50-95=0.131 AR50=0.539 AR50-95=0.289
YOLOv9t      tiny    n= 27346 AP50=0.069 AP50-95=0.024 AR50=0.247 AR50-95=0.092
YOLOv9t      small   n= 23491 AP50=0.187 AP50-95=0.077 AR50=0.541 AR50-95=0.242
YOLOv9t      medium  n= 21856 AP50=0.362 AP50-95=0.202 AR50=0.747 AR50-95=0.425
YOLOv9t      large   n=  2409 AP50=0.469 AP50-95=0.323 AR50=0.856 AR50-95=0.576

Saved:
/kaggle/working/size_results.csv
